# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/G-Yadidhya/notebook/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.** I want to investigate which existing content items an editor or SEO analyst should review first for a possible refresh. This lane fits the starter dataset because it includes search visibility, clicks, engagement, content age, and update-recency signals for many pseudonymized content items. I will start with transparent rules and signal checks before deciding whether a learned model adds value. A high score would mean “worth reviewing,” not “refreshing this page will definitely increase traffic.”


In [1]:
from pathlib import Path
import pandas as pd

# Work locally from the repository root, from the notebook folder, or in Colab.
relative_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/notebook-main/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((path for path in relative_paths if path.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
else:
    # Public, anonymized starter file in this repository (Colab fallback).
    csv_url = "https://raw.githubusercontent.com/G-Yadidhya/notebook/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(csv_url)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Distinct pseudonymized clients: {df['client_id'].nunique():,}")
assert len(df) > 0, "The starter dataset loaded but has no rows."
assert {"content_id", "client_id", "impressions_90d", "ctr", "content_age_days"}.issubset(df.columns)


Rows: 30,000
Columns: 44
Distinct pseudonymized clients: 32


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given the observable search, engagement, content-age, and update-recency signals in the starter snapshot, how can we create a useful shortlist of content items for human review, and does a multi-signal shortlist offer a more useful starting point than sorting by impressions alone?

- **Decision:** Which content item should be reviewed first for a possible refresh?
- **Who acts:** A content editor or SEO analyst reviews the shortlist, checks the page in its real context, and decides whether to update it.
- **Action:** Investigate a page; do not automatically refresh it just because a score is high.
- **Cost of a wrong call:** A false positive wastes editor time on a low-value page. A false negative may leave a genuinely useful improvement opportunity unreviewed.
- **Why data helps:** There are many items and several signals can disagree. A simple transparent rule is the first baseline; a more complex model is justified only if it adds measurable value.
- **Evaluation caution:** This starter file is a snapshot with comparison-window metrics, not a clean independent record of what happened after a recommendation. Before claiming that a ranking identifies successful refreshes, a later phase must define an independent outcome window and prevent leakage.


In [2]:
required_columns = [
    "content_id", "client_id", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "sessions_90d", "engagement_rate", "content_age_days",
    "days_since_last_update", "trend_direction", "trend_pct"
]
missing = [column for column in required_columns if column not in df.columns]
print("Required framing columns present:", not missing)
if missing:
    print("Missing columns:", missing)
assert not missing
print("Rows with unique content IDs:", df["content_id"].nunique() == len(df))
print("Distinct clients for grouped checks:", df["client_id"].nunique())
print("Reminder: IDs are grouping keys only, never model features.")


Required framing columns present: True
Rows with unique content IDs: True
Distinct clients for grouped checks: 32
Reminder: IDs are grouping keys only, never model features.


## 3. Quick look at the data (2–3 real numbers)

The starter CSV contains **30,000 rows**. The code below computes the summary values directly from the file rather than relying on numbers copied from documentation. CTR is stored as a percentage on a 0–100 scale, so a median CTR of `0.07` means **0.07%**, not 7%.


In [3]:
median_impressions = df["impressions_90d"].median()
median_ctr_pct = df["ctr"].median()
down_share_pct = df["trend_direction"].eq("down").mean() * 100

print(f"Median impressions per item over 90 days: {median_impressions:,.0f}")
print(f"Median CTR: {median_ctr_pct:.2f}%")
print(f"Items marked down by the snapshot's trend rule: {down_share_pct:.1f}%")

print("\nInterpretation:")
print("- The median item has modest search visibility, so ranking should consider volume as well as rates.")
print("- CTR values are percentage points on the dataset's documented scale; low-volume rates can be noisy.")
print("- The 'down' share is descriptive only: trend_direction is derived from trend_pct and must not be used as a feature.")


Median impressions per item over 90 days: 731
Median CTR: 0.07%
Items marked down by the snapshot's trend rule: 54.2%

Interpretation:
- The median item has modest search visibility, so ranking should consider volume as well as rates.
- CTR values are percentage points on the dataset's documented scale; low-volume rates can be noisy.
- The 'down' share is descriptive only: trend_direction is derived from trend_pct and must not be used as a feature.


## 4. Careful words: what I can and can't claim

**I can claim:** these are descriptive measurements from this anonymized starter snapshot; the proposed shortlist is decision support for human review; and any later comparison measures performance only on the specific data split and outcome definition tested.

**I cannot claim:** that the score proves a refresh causes growth, that the data represents every website, that a high score guarantees improvement, or that I am “predicting Google’s algorithm.” The `trend_direction` label is derived from `trend_pct`; neither `trend_direction` nor `trend_pct` may be used as a feature when the task is to explain or predict that same trend. `content_id` and `client_id` are pseudonymous identifiers for grouping/validation only. Also, `avg_position = 0` means no position data, not rank zero, and rate columns such as CTR are percentage values on a 0–100 scale.

The next step is to audit signals and define an honest baseline. If I later want to evaluate actual future improvement, I must define a later outcome window and use only features available before that window.


In [4]:
# Basic checks that keep the framing honest before any modeling.
assert df["content_id"].nunique() == len(df), "Expected one row per content item in the starter snapshot."
assert df["trend_direction"].notna().any(), "Expected observed trend categories for descriptive checks."

forbidden_features = {"content_id", "client_id", "trend_direction", "trend_pct"}
print("Do-not-use-as-features list:", sorted(forbidden_features))
print("Observed trend categories (descriptive only):", sorted(df["trend_direction"].dropna().unique()))
print("Framing checks passed. No model has been trained in this assignment.")


Do-not-use-as-features list: ['client_id', 'content_id', 'trend_direction', 'trend_pct']
Observed trend categories (descriptive only): ['down', 'flat', 'new', 'stable', 'up']
Framing checks passed. No model has been trained in this assignment.


## Self-check

- [x] Every section above is filled — markdown thinking and the code that backs it
- [ ] The notebook runs top to bottom with no errors (run **Runtime → Run all** in Colab after opening the notebook)
- [x] No client names, URLs, or private queries were added; the supplied dataset is anonymized
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit the repository URL on the assignment card
